# Engine: Noether Currents

**Claim:** Emmy Noether's theorem applied to L_NN gives conserved currents at each algebra stratum.
Violation of conservation = the system has crossed an algebra boundary.

```
U(1)  → probability current  J^μ = g·Ψ̄·Ψ
SU(2) → isospin current      J^μ_a = g·Ψ̄·T^a·Ψ  (3 components)
SU(3) → colour current       J^μ_a = g·Ψ̄·T^a·Ψ  (8 components)

Violation: ∂_μJ^μ
  < 0.2   → PASS      (conserved)
  0.2–0.5 → MARGINAL
  ≥ 0.5   → VIOLATION (algebra boundary crossed)
```

**File:** `ValaQuenta/modules/noether/maths.py`

The violation diagnostic has no gradient-descent analog. It is a structural test, not a loss function.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../..'))
from ValaQuenta.modules.noether import maths as nt
import math
print('noether module loaded')
ALG_NAMES = {0:'ℝ  (trivial)', 1:'ℂ  U(1)', 2:'ℍ  SU(2)', 3:'𝕆  G₂/SU(3)'}

## Activation Current J^a at Each Algebra Stratum

In [ ]:
psi_norms = [0.9, 0.7, 0.5, 0.4, 0.3, 0.2, 0.15, 0.1]  # 8-component activation
g = 1.0

for alg in [0, 1, 2, 3]:
    J = nt.activation_current(psi_norms[:nt.N_GEN[alg]+1], g, alg)
    print(f'{ALG_NAMES[alg]}: J = {[f"{j:.4f}" for j in J[:3]]}')

## Conservation Violation Diagnostic

In [ ]:
print('Testing conservation across layer transitions:')
print(f'  Threshold PASS:      < {nt.THRESHOLDS["PASS"]}')
print(f'  Threshold MARGINAL:  < {nt.THRESHOLDS["MARGINAL"]}')
print()

for violation_val in [0.05, 0.15, 0.25, 0.45, 0.65]:
    status = nt.conservation_status(violation_val)
    marker = '✓' if status == 'PASS' else ('⚠' if status == 'MARGINAL' else '✗')
    print(f'  ∂_μJ^μ = {violation_val:.2f}  →  {status:10s}  {marker}')

## Blockchain Ledger — Every Violation is a Ledger Entry

In [ ]:
ledger = nt.NoetherLedger()

# Simulate a training sequence across algebra strata
psi_seq = [
    [0.9, 0.8],                          # ℝ — simple
    [0.9, 0.7, 0.5],                     # ℂ — mild violation
    [0.9, 0.6, 0.4, 0.3, 0.25],         # ℍ — moderate
    [0.9, 0.4, 0.3, 0.2, 0.15, 0.1, 0.08, 0.06, 0.04]  # 𝕆 — near boundary
]

for step, psi in enumerate(psi_seq):
    alg = min(step, 3)
    entry = ledger.record(psi_norms=psi, g=1.0, algebra=alg, layer=step+1)
    print(f'Layer {step+1} ({nt.ALG_NAME[alg]}): violation={entry["violation"]:.4f}  {entry["status"]}')

print(f'\nTotal ledger entries: {len(ledger.entries)}')
print(f'Violations recorded:  {sum(1 for e in ledger.entries if e["status"]=="VIOLATION")}')

## Gauge Groups at Each Stratum

In [ ]:
print('Algebra → Gauge Group → N generators')
for alg in [0, 1, 2, 3]:
    print(f'  {nt.ALG_NAME[alg]:2s}  →  {nt.ALG_GAUGE[alg]:12s}  →  {nt.N_GEN[alg]} generators')
print()
print('Total generators: 0+1+3+8 = 12  (U(1)×SU(2)×SU(3) Standard Model gauge group)')